In [2]:
df = spark.read.format('csv') \
.option('header', 'true') \
.option('inferSchema', 'true') \
.load('Files/raw_uploads')
print(f'Rows: {df.count()}') # Should print ~9994
print(f'Columns: {len(df.columns)}')
df.printSchema()

StatementMeta(, fd9aefa9-df23-4b09-978a-001b0887d3b1, 4, Finished, Available, Finished, False)

Rows: 20426
Columns: 21
root
 |-- Row ID: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State/Province: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)



In [4]:
df.show(5, truncate=False)

StatementMeta(, fd9aefa9-df23-4b09-978a-001b0887d3b1, 6, Finished, Available, Finished, False)

+------+--------------+----------+---------+--------------+-----------+-------------+-----------+--------------+------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|Customer Name|    Segment|Country/Region|        City|State/Province|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+---------+--------------+-----------+-------------+-----------+--------------+------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|     1|US-2023-103800|  1/3/2023| 1/7/2023|Standard Class|   DP-13000|Darren Powers|   Consumer| United States|     Houston|         Texas|      77095|Central|OFF-PA-10000174|Office Supplies|       Paper|Message B

In [6]:
df.write.mode('overwrite') \
.parquet('Files/parquet_output/superstore.parquet')
print('Parquet write complete')

StatementMeta(, f5de03ae-36f3-430e-96ef-7ef417cb7663, 8, Finished, Available, Finished, False)

Parquet write complete


In [7]:
print(df.columns)

StatementMeta(, f5de03ae-36f3-430e-96ef-7ef417cb7663, 9, Finished, Available, Finished, False)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [8]:
from pyspark.sql.functions import col

df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

df.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_orders")

print('Delta table created')

StatementMeta(, f5de03ae-36f3-430e-96ef-7ef417cb7663, 10, Finished, Available, Finished, False)

Delta table created


In [2]:
%%sql

SELECT
Region,
COUNT(*) AS total_orders,
ROUND(SUM(Sales),2) AS total_sales
FROM bronze_orders
GROUP BY Region
ORDER BY total_sales DESC

StatementMeta(, d91e14f9-f922-4efc-93e3-15c2c7f1ec12, 3, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 3 fields>

In [11]:

from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, 'bronze_orders')
dt.history().show(truncate=False)

StatementMeta(, 57b6943c-d118-419f-8cc7-413291ef7c71, 13, Finished, Available, Finished, False)

+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------+------------+-------------------------------------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                 |userMetadata|engineInfo                                                   |
+-------+-----------------------+------+--------+---------------------------------

## Day 2 - Delta Mutations & Time Travel

In [4]:
%%sql
SELECT COUNT(*) AS total_rows FROM bronze_orders;

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
%%sql
UPDATE bronze_orders
SET Ship_Mode = 'Priority'
WHERE Category = 'Furniture' AND Region = 'West';

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
%%sql
DELETE FROM bronze_orders
WHERE Sales <= 0;

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [9]:
from pyspark.sql import Row
updates = spark.createDataFrame([
Row(Order_ID='CA-2017-999001', Sales=250.0, Region='West',
Category='Technology'), # new row
Row(Order_ID='CA-2017-999002', Sales=99.5, Region='East', Category='OfficeSupplies'), # new row
])

updates.createOrReplaceTempView('incoming_updates')
updates.show()

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 14, Finished, Available, Finished, False)

+--------------+-----+------+--------------+
|      Order_ID|Sales|Region|      Category|
+--------------+-----+------+--------------+
|CA-2017-999001|250.0|  West|    Technology|
|CA-2017-999002| 99.5|  East|OfficeSupplies|
+--------------+-----+------+--------------+



In [10]:
%%sql
MERGE INTO bronze_orders AS target
USING incoming_updates AS source
ON target.Order_ID = source.Order_ID
WHEN MATCHED THEN
UPDATE SET target.Sales = source.Sales
WHEN NOT MATCHED THEN
INSERT (Order_ID, Sales, Region, Category)
VALUES (source.Order_ID, source.Sales, source.Region, source.Category);

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [11]:
%%sql
SELECT * FROM bronze_orders WHERE Order_ID IN ('CA-2017-999001','CA-2017-999002');

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 17, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 21 fields>

In [12]:
%%sql
DESCRIBE HISTORY bronze_orders;

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 18, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 15 fields>

In [13]:
df_v0 = spark.read.format('delta') \
.option('versionAsOf', 0) \
.table('bronze_orders')
print(f'Version 0 row count: {df_v0.count()}')
df_v0.filter("Order_ID IN ('CA-2017-999001','CA-2017-999002')").show()

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 19, Finished, Available, Finished, False)

Version 0 row count: 10194
+------+--------+----------+---------+---------+-----------+-------------+-------+--------------+----+--------------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country/Region|City|State/Province|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+--------------+----+--------------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+--------------+----+--------------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [14]:
%%sql
SELECT COUNT(*) AS row_count_at_version_0
FROM bronze_orders VERSION AS OF 0;

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [15]:
%%sql

RESTORE TABLE bronze_orders TO VERSION AS OF 0;

StatementMeta(, 272703ba-92ee-4dd7-89ca-373c32435238, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

In [2]:
df = spark.read \
    .option("header","true") \
    .option("multiLine","true") \
    .csv("Files/raw_uploads")

print(len(df.columns))
display(df.limit(5))

StatementMeta(, ae7822ba-60f5-4265-ac71-ad0ca2f70cfc, 5, Finished, Available, Finished, False)

21


SynapseWidget(Synapse.DataFrame, 52051168-15e2-4803-b0dc-04d4559b7c7c)

In [5]:
df = spark.read \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("Files/raw_uploads/superstore_raw.csv")

print(len(df.columns))
display(df.limit(5))

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 8, Finished, Available, Finished, False)

21


SynapseWidget(Synapse.DataFrame, 139c5f44-dcb6-4781-a19e-6cb6db54ef5a)

In [6]:
df = spark.read.text("Files/raw_uploads/superstore_raw.csv")
display(df.limit(20))

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e79f4d3-bd08-4095-b4ba-85fb89477d97)

In [9]:
lines = spark.read.text("Files/raw_uploads/superstore_raw.csv").collect()

for i in range(14,17):
    print(f"Line {i+1}:")
    print(lines[i].value)
    print("-"*100)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 13, Finished, Available, Finished, False)

Line 15:
14,US-2023-167199,1/6/2023,1/10/2023,Standard Class,ME-17320,Maria Etezadi,Home Office,United States,Henderson,Kentucky,42420,South,TEC-PH-10004977,Technology,Phones,GE 30524EE4,391.98,2,0,113.6742
----------------------------------------------------------------------------------------------------
Line 16:
15,US-2023-105417,1/7/2023,1/12/2023,Standard Class,VS-21820,Vivek Sundaresam,Consumer,United States,Huntsville,Texas,77340,Central,FUR-FU-10004864,Furniture,Furnishings,"Howard Miller 14-1/2"" Diameter Chrome Round Wall Clock",76.728,3,0.6,-53.7096
----------------------------------------------------------------------------------------------------
Line 17:
16,US-2023-105417,1/7/2023,1/12/2023,Standard Class,VS-21820,Vivek Sundaresam,Consumer,United States,Huntsville,Texas,77340,Central,OFF-BI-10003708,Office Supplies,Binders,"Acco Four Pocket Poly Ring Binder with Label Holder, Smoke, 1""",10.43,7,0.8,-18.2525
----------------------------------------------------------------

In [10]:
df = spark.read \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("Files/raw_uploads/superstore_raw.csv")

display(df)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ea696bf-b841-4822-9b04-8f57d98d1e1f)

In [11]:
lines = spark.read.text("Files/raw_uploads/superstore_raw.csv").collect()

for i in range(14,17):
    print(f"Line {i+1}:")
    print(lines[i].value)
    print("-"*100)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 15, Finished, Available, Finished, False)

Line 15:
14,US-2023-167199,1/6/2023,1/10/2023,Standard Class,ME-17320,Maria Etezadi,Home Office,United States,Henderson,Kentucky,42420,South,TEC-PH-10004977,Technology,Phones,GE 30524EE4,391.98,2,0,113.6742
----------------------------------------------------------------------------------------------------
Line 16:
15,US-2023-105417,1/7/2023,1/12/2023,Standard Class,VS-21820,Vivek Sundaresam,Consumer,United States,Huntsville,Texas,77340,Central,FUR-FU-10004864,Furniture,Furnishings,"Howard Miller 14-1/2"" Diameter Chrome Round Wall Clock",76.728,3,0.6,-53.7096
----------------------------------------------------------------------------------------------------
Line 17:
16,US-2023-105417,1/7/2023,1/12/2023,Standard Class,VS-21820,Vivek Sundaresam,Consumer,United States,Huntsville,Texas,77340,Central,OFF-BI-10003708,Office Supplies,Binders,"Acco Four Pocket Poly Ring Binder with Label Holder, Smoke, 1""",10.43,7,0.8,-18.2525
----------------------------------------------------------------

In [14]:
df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv("Files/raw_uploads/superstore_raw.csv")

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 19, Finished, Available, Finished, False)

In [15]:
print(df.columns)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 20, Finished, Available, Finished, False)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [17]:

lines = spark.read.text("Files/raw_uploads/superstore_raw.csv")
lines.show(20, truncate=False)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 22, Finished, Available, Finished, False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit     

In [18]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("Files/raw_uploads/superstore_raw.csv")

print(len(df.columns))
print(df.columns)

StatementMeta(, 3d7cb1e9-275d-4f26-a220-d14f05cc9518, 23, Finished, Available, Finished, False)

21
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
